## <span style="color:purple">Morphological synthesis</span>

The reverse operation of morphological analysis is synthesis. 
That is, given the base form (lemma) of the word and some options, generating all possible inflections that match given criteria.

The morphological synthesis interface in EstNLTK 1.7 is same as in previous versions.
There is a function called `synthesize()`, which accepts the following parameters:

 1. word base form (lemma);
 2. word form (see Vabamorf's noun and verb form tables in [00_tables_of_morphological_categories.ipynb](00_tables_of_morphological_categories.ipynb)).
 3. (optional) `partofspeech` (see the POS tag table in [00_tables_of_morphological_categories.ipynb](00_tables_of_morphological_categories.ipynb)).
 4. (optional) `hint`, essentially a prefix filter.
 5. (optional) `guess` (default: True) -- uses guessing for words that are not in Vabamorf's dictionary and cannot be analysed as compounds. Normally, you wouldn't need to change this parameter;
 6. (optional) `phonetic` (default: False) -- adds phonetic markup to synthesized words. Normally, you wouldn't need to change this parameter;

For example, let's generate plural genitive forms for lemma _"palk"_ ( in English: _a paycheck_ and _a log_ ):

In [1]:
from estnltk.vabamorf.morf import synthesize
synthesize('palk', 'pl g')

['palkade', 'palkide']

The result was ambiguous. We can hint the synthesizer so that it outputs only inflections that match prefix _"palka"_ :

In [2]:
synthesize('palk', 'pl g', hint='palka')

['palkade']

Another example.
A demo code for synthesizing all forms of a noun (see Vabamorf's noun form table in [00_tables_of_morphological_categories.ipynb](00_tables_of_morphological_categories.ipynb)) and outputting them in a DataFrame:

In [3]:
from estnltk.vabamorf.morf import synthesize
import pandas

cases = [
    # label, Estonian case name, English case name
    ('n', 'nimetav', 'nominative'),
    ('g', 'omastav', 'genitive'),
    ('p', 'osastav', 'partitive'),
    ('ill', 'sisseütlev', 'illative'),
    ('in', 'seesütlev', 'inessive'),
    ('el', 'seestütlev', 'elative'),
    ('all', 'alaleütlev', 'allative'),
    ('ad', 'alalütlev', 'adessive'),
    ('abl', 'alaltütlev', 'ablative'),
    ('tr', 'saav', 'translative'),
    ('ter', 'rajav', 'terminative'),
    ('es', 'olev', 'essive'),
    ('ab', 'ilmaütlev', 'abessive'),
    ('kom', 'kaasaütlev', 'comitative')]

def synthesize_all(word):
    case_rows = []
    sing_rows = []
    plur_rows = []
    for case, case_name_est, case_name_eng in cases:
        case_rows.append( case_name_est )
        sing_rows.append(', '.join(synthesize(word, 'sg ' + case, 'S')))
        plur_rows.append(', '.join(synthesize(word, 'pl ' + case, 'S')))
    return pandas.DataFrame({'case': case_rows, 'singular': sing_rows, 'plural': plur_rows}, columns=['case', 'singular', 'plural'])

In [4]:
synthesize_all('kuusk')

,case,singular,plural
0,nimetav,kuusk,kuused
1,omastav,kuuse,kuuskede
2,osastav,kuuske,"kuuski, kuuskesid"
3,sisseütlev,kuusesse,kuuskedesse
4,seesütlev,kuuses,kuuskedes
5,seestütlev,kuusest,kuuskedest
6,alaleütlev,kuusele,kuuskedele
7,alalütlev,kuusel,kuuskedel
8,alaltütlev,kuuselt,kuuskedelt
9,saav,kuuseks,kuuskedeks


In [5]:
synthesize_all('luuslang-lendur')

,case,singular,plural
0,nimetav,luuslang-lendur,luuslang-lendurid
1,omastav,luuslang-lenduri,luuslang-lendurite
2,osastav,luuslang-lendurit,luuslang-lendureid
3,sisseütlev,luuslang-lendurisse,"luuslang-lendureisse, luuslang-lenduritesse"
4,seesütlev,luuslang-lenduris,"luuslang-lendureis, luuslang-lendurites"
5,seestütlev,luuslang-lendurist,"luuslang-lendureist, luuslang-lenduritest"
6,alaleütlev,luuslang-lendurile,"luuslang-lendureile, luuslang-lenduritele"
7,alalütlev,luuslang-lenduril,"luuslang-lendureil, luuslang-lenduritel"
8,alaltütlev,luuslang-lendurilt,"luuslang-lendureilt, luuslang-lenduritelt"
9,saav,luuslang-lenduriks,"luuslang-lendureiks, luuslang-lenduriteks"


In [6]:
# A variant of the function that uses English case names instead of Estonian ones
def synthesize_all_eng(word):
    case_rows = []
    sing_rows = []
    plur_rows = []
    for case, case_name_est, case_name_eng in cases:
        case_rows.append( case_name_eng )
        sing_rows.append(', '.join(synthesize(word, 'sg ' + case, 'S')))
        plur_rows.append(', '.join(synthesize(word, 'pl ' + case, 'S')))
    return pandas.DataFrame({'case': case_rows, 'singular': sing_rows, 'plural': plur_rows}, columns=['case', 'singular', 'plural'])

In [7]:
synthesize_all_eng('ratas')

,case,singular,plural
0,nominative,ratas,rattad
1,genitive,ratta,rataste
2,partitive,ratast,rattaid
3,illative,rattasse,"ratastesse, rattaisse"
4,inessive,rattas,"ratastes, rattais"
5,elative,rattast,"ratastest, rattaist"
6,allative,rattale,"ratastele, rattaile"
7,adessive,rattal,"ratastel, rattail"
8,ablative,rattalt,"ratastelt, rattailt"
9,translative,rattaks,"ratasteks, rattaiks"


## <span style="color:purple">Systematic form generators</span>

The function `synthesize()` requires that you know Vabamorf's form labels (such as `'sg g'` or `'ksime'`) and that you assemble larger paradigms yourself. 
For more systematic generation, EstNLTK provides four higher-level functions, which are built on top of `synthesize()`:

 * `generate_noun_forms` -- generates case forms of nouns;
 * `generate_finite_verb_forms` -- generates finite (conjugated) verb forms;
 * `generate_non_finite_verb_forms` -- generates infinitives, participles, gerunds and supines;
 * `generate_verb_phrases` -- generates verb phrases, including multi-word constructions (negation, perfect tenses) and optionally the subject pronoun.

Instead of Vabamorf's form labels, these functions are parameterised with grammatical categories (case, number, voice, mood, tense, person, polarity), using English category names. 
All four functions follow the same conventions:

 * the first argument `lemma` is either a single lemma (string) or a list of lemmas. If a list is given, forms of all lemmas are returned in one flat collection;
 * all grammatical category parameters are optional. **If a category is left unspecified (`None`), all possible values of that category are considered**. So, calling a function with only a lemma generates the whole paradigm covered by the function;
 * the result is a flat collection of strings **in no specific order**, and it does not tell which form corresponds to which category value. If you need a specific form, fix all the categories; if you need a labelled table, call the function once per cell of the table;
 * invalid parameter values and invalid parameter combinations raise a `ValueError`;
 * all additional keyword arguments (e.g. `hint`, `guess`, `phonetic`) are passed on to `synthesize()`. Use them for fine-tuning the synthesis only if you really know what you are doing.

The functions can be imported from `estnltk.vabamorf.morf` (or, alternatively, from `estnltk.vabamorf.form_generators`):

In [1]:
from estnltk.vabamorf.morf import generate_noun_forms
from estnltk.vabamorf.morf import generate_finite_verb_forms
from estnltk.vabamorf.morf import generate_non_finite_verb_forms
from estnltk.vabamorf.morf import generate_verb_phrases

### Generating noun forms

`generate_noun_forms(lemma, case=None, number=None, **kwargs)` generates case forms for the given noun or list of nouns. 
Note that the function works with nouns only (part of speech `'S'`); it does not work with pronouns.

Parameters:

 * `lemma` -- noun lemma (singular nominative) or a list of noun lemmas;
 * `case` -- grammatical case. Possible values:
     * a name of a single case: `'nominative'`, `'genitive'`, `'partitive'`, `'illative'`, `'inessive'`, `'elative'`, `'allative'`, `'adessive'`, `'ablative'`, `'translative'`, `'terminative'`, `'essive'`, `'abessive'`, `'comitative'`;
     * `'short_illative'` or `'long_illative'` -- only one of the two singular illative forms (e.g. _majja_ vs _majasse_). These values can be used only together with `number='singular'`. The plain `'illative'` gives both forms;
     * a wildcard covering a group of cases:
         * `'base_cases'` -- the first three cases: nominative, genitive, partitive;
         * `'location_cases'` -- the six location cases: illative, inessive, elative, allative, adessive, ablative;
         * `'peripheral_cases'` -- the remaining five cases: translative, terminative, essive, abessive, comitative;
         * `'paradigm_cases'` -- the principal forms from which the rest of the paradigm can be derived (singular nominative, genitive, partitive and short illative; plural genitive and partitive). This wildcard already fixes the grammatical number of each form, so `number` must be left unspecified;
     * `None` (default) -- all cases;
 * `number` -- grammatical number: `'singular'`, `'plural'` or `None` (default, both numbers);
 * additional keyword arguments are passed to `synthesize()`.

The result is a tuple of unique word forms.

Let's start by generating the full paradigm of a noun, i.e. all cases in singular and plural:

In [2]:
generate_noun_forms('päkapikk')

('päkapikult',
 'päkapikkudele',
 'päkapikk',
 'päkapikkudeta',
 'päkapikkudena',
 'päkapikusse',
 'päkapiku',
 'päkapikkudeks',
 'päkapikus',
 'päkapikuni',
 'päkapikkudelt',
 'päkapikul',
 'päkapikud',
 'päkapikkudest',
 'päkapikkudeni',
 'päkapikule',
 'päkapikkusid',
 'päkapikuna',
 'päkapikkudes',
 'päkapikkudel',
 'päkapikku',
 'päkapikuga',
 'päkapikuks',
 'päkapikkudesse',
 'päkapikuta',
 'päkapikkudega',
 'päkapikust',
 'päkapikke',
 'päkapikkude')

Fixing both the case and the number gives one specific form (or several forms, if there are parallel forms):

In [3]:
# singular genitive
generate_noun_forms('kuusk', case='genitive', number='singular')

('kuuse',)

In [4]:
# plural partitive
generate_noun_forms('kuusk', case='partitive', number='plural')

('kuuski', 'kuuskesid')

If only one of the categories is fixed, the other one varies over all of its values:

In [5]:
# comitative both in singular and in plural
generate_noun_forms('ratas', case='comitative')

('rattaga', 'ratastega')

In [6]:
# all 14 cases in plural
generate_noun_forms('ratas', number='plural')

('ratasteni',
 'ratastega',
 'ratastelt',
 'ratasteks',
 'rattaini',
 'rattailt',
 'rattaid',
 'ratastest',
 'ratastes',
 'ratastena',
 'ratastesse',
 'ratastel',
 'rattail',
 'rattaile',
 'ratasteta',
 'rattaina',
 'rattad',
 'rattaist',
 'rataste',
 'rattaisse',
 'rattaiks',
 'rattais',
 'ratastele')

Case wildcards select a group of cases at once:

In [8]:
# nominative, genitive and partitive in singular
generate_noun_forms('hiir', case='base_cases', number='singular')

('hiirt', 'hiire', 'hiir')

In [9]:
# the six location cases in singular
generate_noun_forms('maja', case='location_cases', number='singular')

('majas', 'majja', 'majalt', 'majale', 'majast', 'majal', 'majasse')

In [10]:
# principal forms of the paradigm (number must not be specified here)
generate_noun_forms('maja', case='paradigm_cases')

('maju', 'majade', 'majasid', 'maja')

Singular illative often has two parallel forms. You can ask for both of them or pick only one:

In [11]:
print( generate_noun_forms('maja', case='illative', number='singular') )        # both forms
print( generate_noun_forms('maja', case='short_illative', number='singular') )  # short form ("aditive")
print( generate_noun_forms('maja', case='long_illative', number='singular') )   # long form ending with -sse

('majja', 'majasse')
('majja',)
('majasse',)


The lemma can also be a list of lemmas. The result is then a single flat tuple containing the forms of all lemmas:

In [12]:
generate_noun_forms(['kass', 'koer'], case='genitive', number='plural')

('kasside', 'koerade', 'koerte')

Finally, additional keyword arguments are passed to `synthesize()`. 
For instance, the `hint` parameter can be used to disambiguate the forms of _"palk"_, just like in the beginning of this tutorial:

In [13]:
print( generate_noun_forms('palk', case='genitive', number='plural') )
print( generate_noun_forms('palk', case='genitive', number='plural', hint='palka') )

('palkide', 'palkade')
('palkade',)


### Generating finite verb forms

`generate_finite_verb_forms(lemma, voice=None, mood=None, tense=None, person=None, polarity='positive', **kwargs)` generates finite (conjugated) forms for the given verb or list of verbs. 
The function generates single-word forms only.

Parameters:

 * `lemma` -- verb lemma (the _ma_-form, e.g. `'lugema'`) or a list of verb lemmas;
 * `voice` -- `'active'` (personal voice, _isikuline tegumood_), `'passive'` (impersonal voice, _umbisikuline tegumood_) or `None` (default, both);
 * `mood` -- `'indicative'` (_kindel kõneviis_), `'conditional'` (_tingiv kõneviis_), `'imperative'` (_käskiv kõneviis_), `'quotative'` (_kaudne kõneviis_), `'jussive'` (_möönev kõneviis_) or `None` (default, all moods). Note that imperative and jussive have forms only in the present tense, and their forms largely coincide;
 * `tense` -- `'present'`, `'past'` or `None` (default, both). Present perfect and past perfect are not available here, because these are multi-word constructions consisting of a form of the verb _olema_ and a participle. Use `generate_verb_phrases` for the perfect tenses;
 * `person` -- grammatical person and number: `'1s'`, `'2s'`, `'3s'`, `'1p'`, `'2p'`, `'3p'` (e.g. `'1s'` = first person singular, `'3p'` = third person plural) or `None` (default, all persons). **The person can be specified only together with `voice='active'`**, as impersonal forms do not distinguish persons; otherwise a `ValueError` is raised;
 * `polarity` -- `'positive'` (default), `'negative'` or `None` (both). In case of the negative polarity, the function returns only the form of the main verb that is used in the negation construction, without the negation word (e.g. _loe_ for _ei loe_ and _lugenud_ for _ei lugenud_). Strictly speaking, some of these forms are not finite forms. If you need the whole negation phrase, use `generate_verb_phrases`;
 * additional keyword arguments are passed to `synthesize()`.

The result is a list of word forms. If the voice is not specified, the list contains active forms followed by passive forms.

Let's generate all positive finite forms of a verb:

In [14]:
generate_finite_verb_forms('teretama')

['teretanuksin',
 'teretagu',
 'teretage',
 'teretaksid',
 'teretasite',
 'teretaksin',
 'teretaksite',
 'teretas',
 'teretanuksite',
 'teretab',
 'teretaks',
 'teretavad',
 'teretanuksime',
 'teretame',
 'teretate',
 'teretagem',
 'teretasime',
 'teretaksime',
 'teretanuks',
 'teretad',
 'teretanuvat',
 'teretanuksid',
 'teretasid',
 'teretan',
 'teretasin',
 'teretavat',
 'tereta',
 'teretataks',
 'teretatavat',
 'teretatakse',
 'teretatuks',
 'teretati',
 'teretatuvat',
 'teretatagu']

Fixing voice, mood, tense and person pinpoints a single cell of the conjugation table:

In [15]:
# indicative present, 1st person singular  -->  'teretan'
generate_finite_verb_forms('teretama', voice='active', mood='indicative', tense='present', person='1s')

['teretan']

In [16]:
# conditional present, 1st person plural (has two parallel forms: with and without the personal ending)
generate_finite_verb_forms('lugema', voice='active', mood='conditional', tense='present', person='1p')

['loeksime', 'loeks']

Leaving the person unspecified gives the forms of all persons:

In [17]:
# indicative past in all persons
generate_finite_verb_forms('lugema', voice='active', mood='indicative', tense='past')

['lugesite', 'lugesin', 'lugesid', 'luges', 'lugesime']

In [18]:
# imperative forms (the imperative has no 1st person singular form)
generate_finite_verb_forms('lugema', voice='active', mood='imperative')

['loe', 'lugegu', 'lugegem', 'lugege']

Impersonal (passive voice) forms. Remember that the person cannot be specified here:

In [19]:
# indicative present and past in impersonal voice  -->  'teretatakse', 'teretati'
generate_finite_verb_forms('teretama', voice='passive', mood='indicative')

['teretati', 'teretatakse']

Forms of the main verb used in negation. Note that the negation word _ei_ is not part of the output:

In [20]:
print( generate_finite_verb_forms('teretama', voice='active', mood='indicative', tense='present', polarity='negative') )  # (ei) tereta
print( generate_finite_verb_forms('teretama', voice='active', mood='indicative', tense='past', polarity='negative') )     # (ei) teretanud

['tereta']
['teretanud']


A list of lemmas can be conjugated with a single call:

In [21]:
generate_finite_verb_forms(['lugema', 'kirjutama'], voice='active', mood='quotative', tense='present')

['lugevat', 'kirjutavat']

### Generating non-finite verb forms

`generate_non_finite_verb_forms(lemma, voice=None, type=None, tense=None, case=None, **kwargs)` generates non-finite forms for the given verb or list of verbs.

Parameters:

 * `lemma` -- verb lemma or a list of verb lemmas;
 * `voice` -- `'active'`, `'passive'` or `None` (default, both). Infinitive and gerund do not distinguish voice, so leave the voice unspecified when generating these two types;
 * `type` -- type of the non-finite form:
     * `'infinitive'` -- the _da_-infinitive (_lugeda_). By tradition, the _ma_-infinitive is classified as a supine;
     * `'participle'` -- participles: active present (_lugev_), active past (_lugenud_), passive present (_loetav_) and passive past (_loetud_);
     * `'gerund'` -- the _des_-form (_lugedes_);
     * `'supine'` -- the _ma_-infinitive and its case forms in active voice (_lugema_, _lugemaks_, _lugemas_, _lugemast_, _lugemata_) and the impersonal supine (_loetama_);
     * `None` (default) -- all types;
 * `tense` -- `'present'` or `'past'`; selects the subtype of a participle. This parameter is meaningful only if the type is `'participle'` (or unspecified);
 * `case` -- selects the case form of a supine, using Vabamorf's case abbreviations: `'ill'` (_lugema_), `'tr'` (_lugemaks_), `'in'` (_lugemas_), `'el'` (_lugemast_), `'ab'` (_lugemata_). This parameter is meaningful only if the type is `'supine'` (or unspecified) and the voice is `'active'` (or unspecified);
 * additional keyword arguments are passed to `synthesize()`.

The result is a list of unique word forms.

All non-finite forms of a verb:

In [22]:
generate_non_finite_verb_forms('teretama')

['teretatav',
 'teretamas',
 'teretamaks',
 'teretama',
 'teretav',
 'teretatama',
 'teretamata',
 'teretada',
 'teretanud',
 'teretades',
 'teretatud',
 'teretamast']

Infinitive and gerund:

In [23]:
print( generate_non_finite_verb_forms('teretama', type='infinitive') )  # 'teretada'
print( generate_non_finite_verb_forms('teretama', type='gerund') )      # 'teretades'

['teretada']
['teretades']


Participles can be narrowed down by voice and tense:

In [24]:
# all four participles
print( generate_non_finite_verb_forms('teretama', type='participle') )
# past participles of both voices  -->  'teretanud', 'teretatud'
print( generate_non_finite_verb_forms('teretama', type='participle', tense='past') )
# active present participle  -->  'teretav'
print( generate_non_finite_verb_forms('teretama', voice='active', type='participle', tense='present') )

['teretatud', 'teretanud', 'teretatav', 'teretav']
['teretatud', 'teretanud']
['teretav']


Supine forms can be narrowed down by voice and case:

In [25]:
# all supine forms of the active voice
print( generate_non_finite_verb_forms('teretama', voice='active', type='supine') )
# inessive form of the supine  -->  'teretamas'
print( generate_non_finite_verb_forms('teretama', type='supine', case='in') )
# abessive form of the supine  -->  'teretamata'
print( generate_non_finite_verb_forms('teretama', type='supine', case='ab') )
# impersonal supine  -->  'teretatama'
print( generate_non_finite_verb_forms('teretama', voice='passive', type='supine') )

['teretamaks', 'teretama', 'teretamata', 'teretamas', 'teretamast']
['teretamas']
['teretamata']
['teretatama']


And, again, several lemmas can be processed at once:

In [26]:
generate_non_finite_verb_forms(['hängima', 'teretama'], type='infinitive')

['teretada', 'hängida']

### Generating verb phrases

Many cells of the Estonian conjugation table are filled with multi-word constructions rather than with single words: 
negation is expressed with the help of the words _ei_ / _ära_ (_ei loe_, _ära loe_), and the perfect tenses are formed from the verb _olema_ and a past participle (_olen lugenud_, _ei olnud lugenud_). 
`generate_verb_phrases(lemma, voice=None, mood=None, tense=None, person=None, polarity=None, output_type='verb_form', flatten_output=True, **kwargs)` generates such partial or complete verb phrases for the given verb or list of verbs.

Parameters:

 * `lemma` -- verb lemma or a list of verb lemmas;
 * `voice` -- `'active'`, `'passive'` or `None` (default, both);
 * `mood` -- `'indicative'`, `'conditional'`, `'imperative'`, `'quotative'`, `'jussive'` or `None` (default, all moods);
 * `tense` -- `'present'`, `'past'`, `'present_perfect'`, `'past_perfect'` or `None` (default, all tenses). Perfect tenses exist only in indicative, conditional and quotative moods;
 * `person` -- `'1s'`, `'2s'`, `'3s'`, `'1p'`, `'2p'`, `'3p'` or `None` (default, all persons). As in `generate_finite_verb_forms`, the person can be specified only together with `voice='active'`;
 * `polarity` -- `'positive'`, `'negative'` or `None`. Note that, differently from `generate_finite_verb_forms`, the default value is `None`, so by default both affirmative and negative phrases are generated;
 * `output_type` -- determines how much of the phrase is returned:
     * `'verb_form'` (default) -- only the form of the main verb (e.g. _lugenud_);
     * `'verb_phrase'` -- the verb phrase without the subject (e.g. _ei olnud lugenud_);
     * `'complete_phrase'` -- the verb phrase together with the subject pronoun _ma_, _sa_, _ta_, _me_, _te_ or _nad_ (e.g. _ma ei olnud lugenud_). Impersonal (passive voice) phrases have no subject, so for these the complete phrase is the same as the verb phrase;
 * `flatten_output` -- if `True` (default), each phrase is a string in which words are separated by spaces. If `False`, each phrase is a tuple of words. For `output_type='verb_form'` only the flattened output is available (`flatten_output=False` raises a `ValueError`);
 * additional keyword arguments are passed to `synthesize()`.

The result is a list of phrases in no specific order.

With the default `output_type='verb_form'`, the function returns main verb forms, much like `generate_finite_verb_forms`, 
but the output also covers the main verb forms used in negation and in perfect tenses:

In [27]:
generate_verb_phrases('hängima')

['hängib',
 'hängigu',
 'hängi',
 'hängisime',
 'hängiksid',
 'hängiksin',
 'hängisite',
 'hängige',
 'hängiks',
 'hänginuks',
 'hängivad',
 'hängiksite',
 'hängigem',
 'hängisid',
 'hänginuvat',
 'hängid',
 'hänginuksid',
 'hängite',
 'hängivat',
 'hängin',
 'hänginuksin',
 'hänginuksime',
 'hängisin',
 'hängis',
 'hängime',
 'hängiksime',
 'hänginuksite',
 'hänginud',
 'hängitavat',
 'hängiti',
 'hängitaks',
 'hängituvat',
 'hängita',
 'hängituks',
 'hängitud',
 'hängitagu',
 'hängitakse',
 'hängitud',
 'hänginud']

With `output_type='verb_phrase'`, we get phrases without the subject. For instance, let's generate the present perfect phrases of the indicative mood:

In [28]:
# 'olen hänginud', 'oled hänginud', 'on hänginud', ...
generate_verb_phrases('hängima', voice='active', mood='indicative', tense='present_perfect', 
                                 polarity='positive', output_type='verb_phrase')

['oleme hänginud',
 'on hänginud',
 'olete hänginud',
 'olen hänginud',
 'oled hänginud']

Negative phrases. Note that negative perfect tenses have two parallel constructions (e.g. _ei ole hänginud_ and _pole hänginud_), and both of them are generated:

In [29]:
# negation of the indicative present  -->  'ei hängi'
print( generate_verb_phrases('hängima', voice='active', mood='indicative', tense='present', 
                                        polarity='negative', output_type='verb_phrase') )
# negation of the indicative present perfect  -->  'ei ole hänginud', 'pole hänginud'
print( generate_verb_phrases('hängima', voice='active', mood='indicative', tense='present_perfect', 
                                        polarity='negative', output_type='verb_phrase') )
# negation of the imperative  -->  'ära hängi', 'ärgu hängigu', 'ärgem hängigem', 'ärge hängige'
print( generate_verb_phrases('hängima', voice='active', mood='imperative', tense='present', 
                                        polarity='negative', output_type='verb_phrase') )

['ei hängi']
['pole hänginud', 'ei ole hänginud']
['ära hängi', 'ärgem hängigem', 'ärgu hängigu', 'ärge hängige']


With `output_type='complete_phrase'`, phrases are generated together with the matching subject pronoun:

In [30]:
# 'ma hängin', 'sa hängid', 'ta hängib', 'me hängime', 'te hängite', 'nad hängivad'
generate_verb_phrases('hängima', voice='active', mood='indicative', tense='present', 
                                 polarity='positive', output_type='complete_phrase')

['ma hängin',
 'sa hängid',
 'ta hängib',
 'me hängime',
 'te hängite',
 'nad hängivad']

In [31]:
# negative past perfect, 1st person singular  -->  'ma ei olnud lugenud', 'ma polnud lugenud'
generate_verb_phrases('lugema', voice='active', mood='indicative', tense='past_perfect', person='1s', 
                                polarity='negative', output_type='complete_phrase')

['ma polnud lugenud', 'ma ei olnud lugenud']

Impersonal (passive voice) phrases do not have a subject:

In [32]:
# 'on loetud', 'ei ole loetud', 'pole loetud'
generate_verb_phrases('lugema', voice='passive', mood='indicative', tense='present_perfect', 
                                output_type='complete_phrase')

['pole loetud', 'ei ole loetud', 'on loetud']

If you need phrases as sequences of words rather than as strings (e.g. for further processing word by word), set `flatten_output=False`. 
Each phrase is then a tuple of words:

In [33]:
# ('ma', 'ei', 'olnud', 'lugenud'), ('ma', 'polnud', 'lugenud')
generate_verb_phrases('lugema', voice='active', mood='indicative', tense='past_perfect', person='1s', 
                                polarity='negative', output_type='complete_phrase', flatten_output=False)

[('ma', 'ei', 'olnud', 'lugenud'), ('ma', 'polnud', 'lugenud')]

As the output of the generators is unordered, a labelled conjugation table can be assembled by calling the function once for each combination of categories. 
For example, the following code collects the active indicative phrases of a verb into a DataFrame:

In [34]:
import pandas

def conjugation_table(lemma, mood='indicative', polarity='positive'):
    persons = ['1s', '2s', '3s', '1p', '2p', '3p']
    tenses  = ['present', 'past', 'present_perfect', 'past_perfect']
    columns = {'person': persons}
    for tense in tenses:
        columns[tense] = []
        for person in persons:
            phrases = generate_verb_phrases(lemma, voice='active', mood=mood, tense=tense, person=person,
                                                   polarity=polarity, output_type='complete_phrase')
            columns[tense].append( ', '.join(sorted(phrases)) )
    return pandas.DataFrame(columns, columns=['person'] + tenses)

conjugation_table('lugema')

,person,present,past,present_perfect,past_perfect
0,1s,ma loen,ma lugesin,ma olen lugenud,ma olin lugenud
1,2s,sa loed,sa lugesid,sa oled lugenud,sa olid lugenud
2,3s,ta loeb,ta luges,ta on lugenud,ta oli lugenud
3,1p,me loeme,me lugesime,me oleme lugenud,me olime lugenud
4,2p,te loete,te lugesite,te olete lugenud,te olite lugenud
5,3p,nad loevad,nad lugesid,nad on lugenud,nad olid lugenud


In [35]:
conjugation_table('lugema', polarity='negative')

,person,present,past,present_perfect,past_perfect
0,1s,ma ei loe,ma ei lugenud,"ma ei ole lugenud, ma pole lugenud","ma ei olnud lugenud, ma polnud lugenud"
1,2s,sa ei loe,sa ei lugenud,"sa ei ole lugenud, sa pole lugenud","sa ei olnud lugenud, sa polnud lugenud"
2,3s,ta ei loe,ta ei lugenud,"ta ei ole lugenud, ta pole lugenud","ta ei olnud lugenud, ta polnud lugenud"
3,1p,me ei loe,me ei lugenud,"me ei ole lugenud, me pole lugenud","me ei olnud lugenud, me polnud lugenud"
4,2p,te ei loe,te ei lugenud,"te ei ole lugenud, te pole lugenud","te ei olnud lugenud, te polnud lugenud"
5,3p,nad ei loe,nad ei lugenud,"nad ei ole lugenud, nad pole lugenud","nad ei olnud lugenud, nad polnud lugenud"
